In [3]:
import xarray as xr
from satpy.scene import Scene
from pyhdf.SD import SD, SDC 
import pandas as pd
import cartopy.crs as ccrs
import rioxarray
import numpy as np
import matplotlib.pyplot as plt

/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/pyproj/__init__.py:95: UserWarning: pyproj unable to set database path.
  _pyproj_global_context_initialize()


In [3]:
input_file = "/mnt/seviri/L1b/MSG3-SEVI-MSG15-0100-NA-20230728151242.357000000Z-NA.nat"

In [ ]:
from osgeo import gdal

# First gdalwarp command: Reproject and resize
output_file_warp = "/home/sgirtsou/Projects/rs_tools/notebooks/dev/msg/summer_warp_ts_python.tif"

# Reproject and resize
gdal.Warp(
    input_file,
    output_file_warp,
    dstSRS='EPSG:4326',  # Target projection
    width=3712,          # Target width
    height=3712          # Target height
)

# Second gdalwarp command: Cropping
output_file_crop = "/home/sgirtsou/Projects/rs_tools/notebooks/dev/msg/summer_warp_ts_crop_python.tif"

gdal.Warp(
    output_file_warp,
    output_file_crop,
    outputBounds=(-10.019531, 30.22889, 46.617188, 49.012224)  # xmin, ymin, xmax, ymax
)

<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x732d5a89d140> >

In [4]:
msg = rioxarray.open_rasterio('summer_warp_ts_crop.tif')
MODIS_File = '/mnt/data8tb/fire_detection/modis/AF/MOD14.A2023219.0835.061.2023220113415.hdf'

In [5]:
da

NameError: name 'da' is not defined

In [134]:
msg[1,:,:]

<xarray.DataArray (y: 470, x: 1293)> Size: 1MB
[607710 values with dtype=uint16]
Coordinates:
    band         int64 8B 2
  * x            (x) float64 10kB -9.998 -9.954 -9.91 ... 46.51 46.55 46.6
  * y            (y) float64 4kB 48.99 48.95 48.91 48.87 ... 30.33 30.29 30.25
    spatial_ref  int64 8B 0
Attributes: (12/19)
    ch01_cal:                       -1.114656032994e+00 2.185600064695e-02
    ch02_cal:                       -1.465775717050e+00 2.874070033431e-02
    ch03_cal:                       -1.211260244250e+00 2.375020086765e-02
    ch04_cal:                       -1.865920103496e-01 3.658666869601e-03
    ch05_cal:                       -4.242236706827e-01 8.318111189856e-03
    ch06_cal:                       -1.969720376950e+00 3.862196817550e-02
    ...                             ...
    Radiometric parameters format:  offset slope
    AREA_OR_POINT:                  Area
    _FillValue:                     0
    scale_factor:                   1.0
    add_offset:                     0.0
    long_name:                      ('band 01', 'band 02', 'band 03', 'band 0...

In [6]:
msg_sub = msg[1,:,:]
msg_sub = msg_sub.drop_attrs()
msg_sub.rio.write_crs('EPSG:4326', inplace=True)
msg_sub = msg_sub.rio.to_raster('msg_sub.tif')

In [7]:
#gdalwarp -t_srs EPSG:4326 -ts 3712 3712 /mnt/seviri/L1b/MSG3-SEVI-MSG15-0100-NA-20230728151242.357000000Z-NA.nat /home/sgirtsou/Projects/rs_tools/notebooks/dev/msg/summer_warp_ts.tif
#gdalwarp -te -10.019531, 30.22889, 46.617188, 49.012224 summer_warp_ts.tif summer_warp_ts_crop.tif

In [8]:
f = SD(MODIS_File, SDC.READ)
df = pd.DataFrame()
sds_obj = f.select('fire mask') # select sds
fm_data = sds_obj.get() # get sds data

if fm_data[fm_data > 6].shape[0] > 0:

    for key in ['FP_latitude', 
                    'FP_longitude', 
                    'FP_ViewZenAng',
                    'FP_power']:

        sds_obj = f.select(key) # select sds
        
        data = sds_obj.get()#
        df[key] = data.ravel()

In [13]:
key

'FP_power'

In [53]:
df

,FP_latitude,FP_longitude,FP_ViewZenAng,FP_power
0,54.449600,41.659851,39.430000,19.840338
1,54.439064,41.653038,39.430000,15.837902
2,54.582291,39.895741,32.340000,8.146684
3,54.196159,37.387146,22.160000,7.008280
4,55.106518,30.440529,17.779999,16.187088
...,...,...,...,...
121,36.802258,39.963196,64.470001,114.596985
122,36.817623,39.964649,64.470001,112.653816
123,36.800999,39.957680,64.470001,109.354004
124,38.620953,29.478470,15.960000,7.316031


In [133]:
msg

<xarray.DataArray (band: 11, y: 470, x: 1293)> Size: 13MB
[6684810 values with dtype=uint16]
Coordinates:
  * band         (band) int64 88B 1 2 3 4 5 6 7 8 9 10 11
  * x            (x) float64 10kB -9.998 -9.954 -9.91 ... 46.51 46.55 46.6
  * y            (y) float64 4kB 48.99 48.95 48.91 48.87 ... 30.33 30.29 30.25
    spatial_ref  int64 8B 0
Attributes: (12/19)
    ch01_cal:                       -1.114656032994e+00 2.185600064695e-02
    ch02_cal:                       -1.465775717050e+00 2.874070033431e-02
    ch03_cal:                       -1.211260244250e+00 2.375020086765e-02
    ch04_cal:                       -1.865920103496e-01 3.658666869601e-03
    ch05_cal:                       -4.242236706827e-01 8.318111189856e-03
    ch06_cal:                       -1.969720376950e+00 3.862196817550e-02
    ...                             ...
    Radiometric parameters format:  offset slope
    AREA_OR_POINT:                  Area
    _FillValue:                     0
    scale_factor:                   1.0
    add_offset:                     0.0
    long_name:                      ('band 01', 'band 02', 'band 03', 'band 0...

In [57]:
x_size = len(msg.x)
y_size = len(msg.y)

array = np.zeros((y_size, x_size))

In [58]:
for lat, lon, fire_index in zip(latitudes, longitudes, fire_indices):
    print(lat, lon, fire_index)
   # Select the index of the nearest point for the given coordinates
    selected = msg.sel(x=lon, y=lat, method='nearest')
    # Get the indices of the nearest point
    x_idx = msg.get_index('x').get_loc(selected['x'].item())
    y_idx = msg.get_index('y').get_loc(selected['y'].item())
    array[y_idx, x_idx] = 1

54.4496 41.65985 [False False False ... False False False]
54.439064 41.653038 [False False False ... False False False]
54.58229 39.89574 [False False False ... False False False]
54.19616 37.387146 [False False False ... False False False]
55.106518 30.440529 [False False False ... False False False]
55.10151 30.432024 [False False False ... False False False]
55.099483 30.448921 [False False False ... False False False]
54.56513 33.168568 [False False False ... False False False]
53.05078 38.510616 [False False False ... False False False]
52.537052 39.62327 [False False False ... False False False]
49.714466 37.973022 [False False False ... False False False]
50.188747 35.37139 [False False False ... False False False]
50.05152 34.892952 [False False False ... False False False]
49.3397 37.94241 [False False False ... False False False]
48.872498 38.405018 [False False False ... False False False]
49.281265 34.76626 [False False False ... False False False]
48.58253 37.75188 [False

In [94]:
array.shape

(470, 1293)

In [95]:
msg

<xarray.DataArray (band: 11, y: 470, x: 1293)> Size: 13MB
[6684810 values with dtype=uint16]
Coordinates:
  * band         (band) int64 88B 1 2 3 4 5 6 7 8 9 10 11
  * x            (x) float64 10kB -9.998 -9.954 -9.91 ... 46.51 46.55 46.6
  * y            (y) float64 4kB 48.99 48.95 48.91 48.87 ... 30.33 30.29 30.25
    spatial_ref  int64 8B 0
Attributes: (12/19)
    ch01_cal:                       -1.114656032994e+00 2.185600064695e-02
    ch02_cal:                       -1.465775717050e+00 2.874070033431e-02
    ch03_cal:                       -1.211260244250e+00 2.375020086765e-02
    ch04_cal:                       -1.865920103496e-01 3.658666869601e-03
    ch05_cal:                       -4.242236706827e-01 8.318111189856e-03
    ch06_cal:                       -1.969720376950e+00 3.862196817550e-02
    ...                             ...
    Radiometric parameters format:  offset slope
    AREA_OR_POINT:                  Area
    _FillValue:                     0
    scale_factor:                   1.0
    add_offset:                     0.0
    long_name:                      ('band 01', 'band 02', 'band 03', 'band 0...

In [128]:
da = xr.DataArray(
    array,
    coords={"y": msg.coords["y"], "x": msg.coords["x"]},
    dims=("y", "x")
)

In [129]:
da = da.expand_dims(band=[12])

In [131]:
da.rio.to_raster('fire_band12.tif')

In [114]:
np.unique(da)

array([0.])

In [99]:
merged = xr.concat([msg, da], dim="band")

In [112]:
np.unique(merged[-1,:,:])

array([0.])

In [107]:
merged.attrs['long_name'] = merged.attrs['long_name'] + ('fire',)

In [108]:
merged.rio.to_raster("summer_warp_ts_fire_test2.tif")

In [91]:
msg_ds = msg.to_dataset(name="Rad")
da_ds = da.to_dataset(name="fire_mask")
merged = xr.merge([msg_ds,da_ds])

In [92]:
merged

<xarray.Dataset> Size: 18MB
Dimensions:      (band: 11, x: 1293, y: 470)
Coordinates:
  * band         (band) int64 88B 1 2 3 4 5 6 7 8 9 10 11
  * x            (x) float64 10kB -9.998 -9.954 -9.91 ... 46.51 46.55 46.6
  * y            (y) float64 4kB 48.99 48.95 48.91 48.87 ... 30.33 30.29 30.25
    spatial_ref  int64 8B 0
Data variables:
    Rad          (band, y, x) uint16 13MB ...
    fire_mask    (y, x) float64 5MB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0

In [81]:
merged = merged.drop_vars('spatial_ref')

ValueError: These variables cannot be found in this dataset: ['spatial_ref']

In [88]:
merged

<xarray.Dataset> Size: 29MB
Dimensions:    (band: 11, y: 470, x: 1293)
Coordinates:
  * x          (x) float64 10kB -9.998 -9.954 -9.91 -9.866 ... 46.51 46.55 46.6
  * y          (y) float64 4kB 48.99 48.95 48.91 48.87 ... 30.33 30.29 30.25
  * band       (band) int64 88B 1 2 3 4 5 6 7 8 9 10 11
Data variables:
    Rad        (band, y, x) float32 27MB 399.0 380.0 376.0 ... 675.0 680.0 677.0
    fire_mask  (y, x) float32 2MB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0 0.0

In [89]:
merged.rio.to_raster('merged.tif')

TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [62]:
da.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
da.rio.write_crs("EPSG:4326", inplace=True)

# Save to GeoTIFF
da.rio.to_raster("fire_mask.tif")

## Test if i can select lat lon from rs_tools geoprocessed

In [ ]:
'20230821101241'

In [3]:
ds = xr.open_dataset('/mnt/outputs/geoprocessed/20230728151242_msg.nc')

In [4]:
raw_msg= ['/mnt/seviri/L1b/MSG3-SEVI-MSG15-0100-NA-20230821101241.735000000Z-NA.nat']

In [5]:
scn = Scene(
            reader="seviri_l1b_native",
            filenames=raw_msg
        )

In [6]:
channels = [x for x in scn.available_dataset_names() if x!='HRV']
assert len(channels) == 11, "Number of channels is not 11"

scn.load(channels, generate=False, calibration='radiance')

In [7]:
from pyresample import create_area_def

area_def = create_area_def('my_area',
                           {'proj': 'longlat', 'datum': 'WGS84'},
                           area_extent=[-10.019531, 30.22889, 46.617188, 49.012224],
                           resolution=0.027,
                           units='degrees',
                           description='Gloxx   bal degree lat-lon grid')

Rounding shape to (696, 2098) and resolution from (0.027000000000001023, 0.027000000000001023) meters to (0.02699557626310772, 0.02698754885057472) meters


In [8]:
new_scn = scn.resample(area_def)

In [9]:
ds_resampled = new_scn.to_xarray_dataset()

In [10]:
ds_resampled = ds_resampled.reset_coords(drop=True)
attrs_dict = {x: ds_resampled[x].attrs for x in channels}

In [11]:
ds_resampled.rio.write_crs('EPSG:4326', inplace=True)
ds_resampled.rio.to_raster('20230821101241_wgs84.tif')

/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/dask/_task_spec.py:745: RuntimeWarning: invalid value encountered in sin
  return self.func(*new_argspec)
/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/dask/_task_spec.py:745: RuntimeWarning: invalid value encountered in cos
  return self.func(*new_argspec)


In [12]:
ds_resampled = ds_resampled.assign(Rad=xr.concat(list(map(lambda x: ds_resampled[x], channels)), dim="band"))
ds_resampled['Rad'] = ds_resampled['Rad'].astype('float32')

In [13]:
ds_resampled = ds_resampled.assign_coords(band=list(map(lambda x: x, channels)))

In [14]:
# drop variables that will no longer be needed
ds_resampled = ds_resampled.drop_vars(list(map(lambda x: x, channels)))

In [15]:
time_stamp = attrs_dict[list(attrs_dict.keys())[0]]['start_time']
# assign bands and time data to each variable
ds_resampled = ds_resampled.assign_coords({"time": [time_stamp]})

In [16]:
MSG_WAVELENGTHS = {
    'IR_016': 1.64,
    'IR_039': 3.92,
    'IR_087': 8.70,
    'IR_097': 9.66,
    'IR_108': 10.80,
    'IR_120': 12.00,
    'IR_134': 13.40,
    'VIS006': 0.64,
    'VIS008': 0.81,
    'WV_062': 6.25,
    'WV_073': 7.35
}

In [17]:
ds_resampled.attrs = {}
ds_resampled.attrs = dict(
    calibration=attrs_dict[list(attrs_dict.keys())[0]]["calibration"],
    standard_name=attrs_dict[list(attrs_dict.keys())[0]]["standard_name"],
    platform_name=attrs_dict[list(attrs_dict.keys())[0]]["platform_name"],
    sensor=attrs_dict[list(attrs_dict.keys())[0]]["sensor"],
    units=attrs_dict[list(attrs_dict.keys())[0]]["units"],
    orbital_parameters=attrs_dict[list(attrs_dict.keys())[0]]["orbital_parameters"]
)
# assign band wavelengths 
ds_resampled = ds_resampled.assign_coords({"band_wavelength": list(MSG_WAVELENGTHS.values())}) 
# change to float32 to reduce file size
ds_resampled['Rad'] = ds_resampled['Rad'].astype('float32')

In [18]:
for var in ds_resampled.data_vars:
    ds_resampled[var].attrs.pop('grid_mapping', None)
    ds_resampled[var].attrs.pop('orbital_parameters', None)
    ds_resampled[var].attrs.pop('georef_offset_corrected', None)
    ds_resampled[var].attrs.pop('time_parameters', None)
    ds_resampled[var].attrs.pop('start_time', None)
    ds_resampled[var].attrs.pop('end_time', None)
    ds_resampled[var].attrs.pop('area', None)
    ds_resampled[var].attrs.pop('_satpy_id', None)
del ds_resampled.attrs['orbital_parameters']
#del ds_resampled.attrs['georef_offset_corrected']

In [19]:
ds_resampled.Rad

<xarray.DataArray 'Rad' (band: 11, y: 696, x: 2098)> Size: 64MB
dask.array<concatenate, shape=(11, 696, 2098), dtype=float32, chunksize=(1, 696, 2098), chunktype=numpy.ndarray>
Coordinates:
  * y            (y) float64 6kB 49.0 48.97 48.94 48.92 ... 30.3 30.27 30.24
  * x            (x) float64 17kB -10.01 -9.979 -9.952 ... 46.55 46.58 46.6
    spatial_ref  int64 8B 0
  * band         (band) <U6 264B 'IR_016' 'IR_039' ... 'WV_062' 'WV_073'
Attributes:
    units:                mW m-2 sr-1 (cm-1)-1
    wavelength:           1.64 µm (1.5-1.78 µm)
    standard_name:        toa_outgoing_radiance_per_unit_wavenumber
    platform_name:        Meteosat-10
    sensor:               seviri
    reader:               seviri_l1b_native
    name:                 IR_016
    resolution:           3000.403165817
    calibration:          radiance
    modifiers:            ()
    ancillary_variables:  []

In [20]:
ds_resampled.to_netcdf('20230821101241_wgs84.nc', engine="netcdf4")

/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/dask/_task_spec.py:745: RuntimeWarning: invalid value encountered in cos
  return self.func(*new_argspec)


/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/dask/_task_spec.py:745: RuntimeWarning: invalid value encountered in sin
  return self.func(*new_argspec)


### test the fire brigade file overlap

In [24]:
ds_resampled

<xarray.Dataset> Size: 64MB
Dimensions:          (y: 696, x: 2098, band: 11, time: 1, band_wavelength: 11)
Coordinates:
  * y                (y) float64 6kB 49.0 48.97 48.94 48.92 ... 30.3 30.27 30.24
  * x                (x) float64 17kB -10.01 -9.979 -9.952 ... 46.55 46.58 46.6
    spatial_ref      int64 8B 0
  * band             (band) <U6 264B 'IR_016' 'IR_039' ... 'WV_062' 'WV_073'
  * time             (time) datetime64[ns] 8B 2023-08-21T10:00:00
  * band_wavelength  (band_wavelength) float64 88B 1.64 3.92 8.7 ... 6.25 7.35
Data variables:
    Rad              (band, y, x) float32 64MB dask.array<chunksize=(1, 696, 2098), meta=np.ndarray>
Attributes:
    calibration:    radiance
    standard_name:  toa_outgoing_radiance_per_unit_wavenumber
    platform_name:  Meteosat-10
    sensor:         seviri
    units:          mW m-2 sr-1 (cm-1)-1

In [27]:
x, y = 26.175290, 41.115503	
selected = ds_resampled.sel(x=x, y=y, method='nearest')
# Get the indices of the nearest point
x_idx = ds_resampled.get_index('x').get_loc(selected['x'].item())
y_idx = ds_resampled.get_index('y').get_loc(selected['y'].item())
array[y_idx, x_idx] = 1
da = xr.DataArray(
        array,
        coords={"y": ds_resampled.y, "x": ds_resampled.x},
        dims=("y", "x")
    )
da.to_netcdf('20230821101241_wgs84_fire_mask.nc')

### end of test 

In [84]:
ds_resampled

<xarray.Dataset> Size: 64MB
Dimensions:          (y: 696, x: 2098, band: 11, time: 1, band_wavelength: 11)
Coordinates:
  * y                (y) float64 6kB 49.0 48.97 48.94 48.92 ... 30.3 30.27 30.24
  * x                (x) float64 17kB -10.01 -9.979 -9.952 ... 46.55 46.58 46.6
    spatial_ref      int64 8B 0
  * band             (band) <U6 264B 'IR_016' 'IR_039' ... 'WV_062' 'WV_073'
  * time             (time) datetime64[ns] 8B 2023-07-28T15:00:00
  * band_wavelength  (band_wavelength) float64 88B 1.64 3.92 8.7 ... 6.25 7.35
Data variables:
    Rad              (band, y, x) float32 64MB dask.array<chunksize=(1, 696, 2098), meta=np.ndarray>
Attributes:
    calibration:    radiance
    standard_name:  toa_outgoing_radiance_per_unit_wavenumber
    platform_name:  Meteosat-10
    sensor:         seviri
    units:          mW m-2 sr-1 (cm-1)-1

In [21]:
array = np.zeros((ds_resampled.y.size, ds_resampled.x.size))

In [95]:
for lat, lon, fire_index in zip(latitudes, longitudes, fire_indices):
    print(lat, lon, fire_index)
   # Select the index of the nearest point for the given coordinates
    selected = msg.sel(x=lon, y=lat, method='nearest')
    # Get the indices of the nearest point
    x_idx = msg.get_index('x').get_loc(selected['x'].item())
    y_idx = msg.get_index('y').get_loc(selected['y'].item())
    array[y_idx, x_idx] = 1

54.4496 41.65985 [False False False ... False False False]
54.439064 41.653038 [False False False ... False False False]
54.58229 39.89574 [False False False ... False False False]
54.19616 37.387146 [False False False ... False False False]
55.106518 30.440529 [False False False ... False False False]
55.10151 30.432024 [False False False ... False False False]
55.099483 30.448921 [False False False ... False False False]
54.56513 33.168568 [False False False ... False False False]
53.05078 38.510616 [False False False ... False False False]
52.537052 39.62327 [False False False ... False False False]
49.714466 37.973022 [False False False ... False False False]
50.188747 35.37139 [False False False ... False False False]
50.05152 34.892952 [False False False ... False False False]
49.3397 37.94241 [False False False ... False False False]
48.872498 38.405018 [False False False ... False False False]
49.281265 34.76626 [False False False ... False False False]
48.58253 37.75188 [False

In [97]:
da = xr.DataArray(
    array,
    coords={"y": ds_resampled.y, "x": ds_resampled.x},
    dims=("y", "x")
)

In [98]:
da.to_netcdf('fire_mask_new.nc')

In [108]:
ds_ = scn.to_xarray()

/home/sgirtsou/miniconda3/envs/rs_tools/lib/python3.11/site-packages/satpy/cf/coords.py:201: UserWarning: Cannot pretty-format "acq_time" coordinates because they are not identical among the given datasets
  _warn_if_pretty_but_not_unique(pretty, coord_name)


In [109]:
var = "msg_seviri_fes_3km"
crs_wkt = ds_[var].crs_wkt

In [110]:
crs_wkt

'PROJCRS["unknown",BASEGEOGCRS["unknown",DATUM["unknown",ELLIPSOID["unknown",6378169,295.488065897014,LENGTHUNIT["metre",1,ID["EPSG",9001]]]],PRIMEM["Greenwich",0,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8901]]],CONVERSION["unknown",METHOD["Geostationary Satellite (Sweep Y)"],PARAMETER["Longitude of natural origin",0,ANGLEUNIT["degree",0.0174532925199433],ID["EPSG",8802]],PARAMETER["Satellite Height",35785831,LENGTHUNIT["metre",1,ID["EPSG",9001]]],PARAMETER["False easting",0,LENGTHUNIT["metre",1],ID["EPSG",8806]],PARAMETER["False northing",0,LENGTHUNIT["metre",1],ID["EPSG",8807]]],CS[Cartesian,2],AXIS["(E)",east,ORDER[1],LENGTHUNIT["metre",1,ID["EPSG",9001]]],AXIS["(N)",north,ORDER[2],LENGTHUNIT["metre",1,ID["EPSG",9001]]]]'

In [113]:
from pyproj import CRS

cc = CRS(crs_wkt)

# assign CRS to dataarray
ds_.rio.write_crs(cc, inplace=True)

<xarray.Dataset> Size: 827MB
Dimensions:             (y: 3712, x: 3712)
Coordinates: (12/16)
    msg_seviri_fes_3km  int64 8B 0
    IR_016_acq_time     (y) datetime64[ns] 30kB NaT NaT NaT NaT ... NaT NaT NaT
  * y                   (y) float64 30kB -5.566e+06 -5.563e+06 ... 5.569e+06
  * x                   (x) float64 30kB 5.566e+06 5.563e+06 ... -5.569e+06
    longitude           (y, x) float64 110MB dask.array<chunksize=(460, 3712), meta=np.ndarray>
    latitude            (y, x) float64 110MB dask.array<chunksize=(460, 3712), meta=np.ndarray>
    ...                  ...
    IR_120_acq_time     (y) datetime64[ns] 30kB NaT NaT NaT NaT ... NaT NaT NaT
    IR_134_acq_time     (y) datetime64[ns] 30kB NaT NaT NaT NaT ... NaT NaT NaT
    VIS006_acq_time     (y) datetime64[ns] 30kB NaT NaT NaT NaT ... NaT NaT NaT
    VIS008_acq_time     (y) datetime64[ns] 30kB NaT NaT NaT NaT ... NaT NaT NaT
    WV_062_acq_time     (y) datetime64[ns] 30kB NaT NaT NaT NaT ... NaT NaT NaT
    WV_073_acq_time     (y) datetime64[ns] 30kB NaT NaT NaT NaT ... NaT NaT NaT
Data variables:
    IR_016              (y, x) float32 55MB dask.array<chunksize=(460, 3712), meta=np.ndarray>
    IR_039              (y, x) float32 55MB dask.array<chunksize=(460, 3712), meta=np.ndarray>
    IR_087              (y, x) float32 55MB dask.array<chunksize=(460, 3712), meta=np.ndarray>
    IR_097              (y, x) float32 55MB dask.array<chunksize=(460, 3712), meta=np.ndarray>
    IR_108              (y, x) float32 55MB dask.array<chunksize=(460, 3712), meta=np.ndarray>
    IR_120              (y, x) float32 55MB dask.array<chunksize=(460, 3712), meta=np.ndarray>
    IR_134              (y, x) float32 55MB dask.array<chunksize=(460, 3712), meta=np.ndarray>
    VIS006              (y, x) float32 55MB dask.array<chunksize=(460, 3712), meta=np.ndarray>
    VIS008              (y, x) float32 55MB dask.array<chunksize=(460, 3712), meta=np.ndarray>
    WV_062              (y, x) float32 55MB dask.array<chunksize=(460, 3712), meta=np.ndarray>
    WV_073              (y, x) float32 55MB dask.array<chunksize=(460, 3712), meta=np.ndarray>
Attributes:
    history:      Created by pytroll/satpy on 2025-02-11 15:20:21.999911
    Conventions:  CF-1.7

In [121]:
from pyproj import CRS, Transformer
import xarray as xr

# TODO: To be moved to Earth System Datacube Tools
def convert_lat_lon_to_x_y(crs, lon, lat):
    transformer = Transformer.from_crs(CRS("+proj=latlon"), crs, always_xy=True)
    x, y = transformer.transform(lon, lat)
    return x, y

In [122]:
convert_lat_lon_to_x_y(cc, 30.22889, -10.019531)

(3073049.3154671057, -1069508.472962889)

In [134]:
ds.sel(x = -930124.472962889 , y=3108417.3154671057, method='nearest').Rad.values

array([  0.1900016,   0.6183147,  59.443085 ,  50.421043 ,  97.80202  ,
       113.15637  ,  96.92824  ,   0.5464   ,   0.4023698,   4.6747785,
        19.697205 ], dtype=float32)

In [ ]:
import os
for file in os.listdir('/mnt/data8tb/fire_detection'):
    file_name = 

In [ ]:
ds.x 2023-06-01 MOD14.A2023157.1210.061.2023157212510.hdf

<xarray.DataArray 'x' (x: 1235)> Size: 10kB
array([-933125.264526, -930124.86145 , -927124.458374, ..., 2763371.325317,
       2766371.728394, 2769372.13147 ])
Coordinates:
  * x            (x) float64 10kB -9.331e+05 -9.301e+05 ... 2.766e+06 2.769e+06
    spatial_ref  int64 8B 0
Attributes:
    units:          m
    standard_name:  projection_x_coordinate